## Data Mode Configuration

Three data modes available:
- `retailrocket`: Full dataset (2.7M events) for production training
- `synthetic`: Simulator-generated data for pipeline testing
- `merged`: Small combined dataset for quick iteration

---

In [5]:
# Configuration: Set Data Mode
# Options: "retailrocket", "synthetic", "merged"
DATA_MODE = "retailrocket"

print(f"Data Mode: {DATA_MODE.upper()}")
print(f"Features will be engineered from {DATA_MODE} dataset")
print(f"Output will be saved to: artifacts/features/{DATA_MODE}/")

Data Mode: RETAILROCKET
Features will be engineered from retailrocket dataset
Output will be saved to: artifacts/features/retailrocket/


---

## 1. Imports & Setup

In [6]:
# Core data processing
import pandas as pd
import numpy as np

# File handling
from pathlib import Path
import json

# Datetime utilities
from datetime import datetime, timezone

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Environment setup complete")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Environment setup complete
pandas: 2.3.3
numpy: 2.3.5


---

## 2. Load Dataset

Load events based on configured DATA_MODE.

In [7]:
from pathlib import Path
import pandas as pd

# Define data source paths based on mode
if DATA_MODE == "retailrocket":
    DATA_PATH = Path("artifacts/external/retailrocket/events.parquet")
    print("Loading RetailRocket dataset (full production data)")

elif DATA_MODE == "synthetic":
    # Load all synthetic parquet files
    DATA_PATH = Path("artifacts/events")
    print("Loading Synthetic dataset (simulator-generated)")

elif DATA_MODE == "merged":
    DATA_PATH = Path("artifacts/combined/events.parquet")
    print("Loading Merged dataset (demo/testing)")

else:
    raise ValueError(
        f"Invalid DATA_MODE: {DATA_MODE}. Use 'retailrocket', 'synthetic', or 'merged'"
    )

# Load dataset
print(f"Path: {DATA_PATH}")

if DATA_MODE == "synthetic":
    # Load all parquet files from synthetic directory
    parquet_files = list(DATA_PATH.rglob("*.parquet"))

    if not parquet_files:
        raise FileNotFoundError(f"Error: No parquet files found in {DATA_PATH}")

    print(f"Found {len(parquet_files)} parquet files")

    df_events = pd.concat(
        [pd.read_parquet(f) for f in parquet_files],
        ignore_index=True
    )

else:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Error: Dataset not found: {DATA_PATH}")

    df_events = pd.read_parquet(DATA_PATH)

print(f"\nLoaded {DATA_MODE} dataset")
print(f"Shape: {df_events.shape}")

print("\nDataset Stats:")
print(f"Total events: {len(df_events):,}")
print(f"Unique users: {df_events['user_id'].nunique():,}")
print(f"Unique products: {df_events['product_id'].nunique():,}")
print(f"Unique sessions: {df_events['session_id'].nunique():,}")

print("\nEvent type distribution:")
print(df_events["event_type"].value_counts())

if "source" in df_events.columns:
    print("\nSource distribution:")
    print(df_events["source"].value_counts())

Loading RetailRocket dataset (full production data)
Path: artifacts\external\retailrocket\events.parquet

Loaded retailrocket dataset
Shape: (2756101, 7)

Dataset Stats:
Total events: 2,756,101
Unique users: 1,407,580
Unique products: 235,061
Unique sessions: 1,666,974

Event type distribution:
event_type
view           2664312
add_to_cart      69332
purchase         22457
Name: count, dtype: int64


In [8]:
# Parse timestamp as datetime with timezone
print("Parsing timestamps...")
df_events['ts_datetime'] = pd.to_datetime(df_events['ts'], format='ISO8601', utc=True)

print("Sorting events by timestamp...")
df_events = df_events.sort_values('ts_datetime').reset_index(drop=True)

print(f"Timestamp processing complete")
print(f"Earliest event: {df_events['ts_datetime'].min()}")
print(f"Latest event: {df_events['ts_datetime'].max()}")
print(f"Time span: {(df_events['ts_datetime'].max() - df_events['ts_datetime'].min()).days} days")

Parsing timestamps...
Sorting events by timestamp...
Timestamp processing complete
Earliest event: 2015-05-03 03:00:04+00:00
Latest event: 2015-09-18 02:59:47+00:00
Time span: 137 days


---

## 3. Define Reference Time

All features are computed relative to a reference time to prevent data leakage. In production, this would be the current timestamp. For this notebook, we use the latest event timestamp as the reference.

In [9]:
# Define reference time (latest event timestamp)
REFERENCE_TIME = df_events['ts_datetime'].max()

print(f"Reference Time: {REFERENCE_TIME}")
print(f"\nAll features will be computed as of this reference time.")
print(f"Recency features measure time elapsed from each event to {REFERENCE_TIME}.")
print(f"\nThis prevents data leakage—features do not use future information.")

Reference Time: 2015-09-18 02:59:47+00:00

All features will be computed as of this reference time.
Recency features measure time elapsed from each event to 2015-09-18 02:59:47+00:00.

This prevents data leakage—features do not use future information.


---

## 4. User-Level Feature Engineering

Create one row per user with aggregate features capturing their behavior patterns.

In [10]:
print("Engineering user-level features\n")

# Basic aggregations per user
user_features = df_events.groupby('user_id').agg(
    total_events=('event_id', 'count'),
    unique_products_interacted=('product_id', 'nunique'),
    unique_sessions=('session_id', 'nunique'),
    last_event_ts=('ts_datetime', 'max')
).reset_index()

print(f"Basic user aggregations computed: {user_features.shape}")

# Event type counts per user
# Create pivot table: rows=user_id, columns=event_type, values=count
event_type_counts = df_events.groupby(['user_id', 'event_type']).size().unstack(fill_value=0)
event_type_counts = event_type_counts.reset_index()

# Rename columns to match requirements
column_mapping = {
    'view': 'views_count',
    'add_to_cart': 'add_to_cart_count',
    'purchase': 'purchase_count'
}

# Rename existing columns
event_type_counts = event_type_counts.rename(columns=column_mapping)

# Ensure all required event type columns exist (fill missing with 0)
for col in ['views_count', 'add_to_cart_count', 'purchase_count']:
    if col not in event_type_counts.columns:
        event_type_counts[col] = 0

# Also handle 'click' if it exists
if 'click' in event_type_counts.columns:
    event_type_counts = event_type_counts.drop(columns=['click'])

print(f"Event type counts computed")

# Merge event type counts with user features
user_features = user_features.merge(event_type_counts, on='user_id', how='left')

# Fill any NaN values with 0
user_features['views_count'] = user_features['views_count'].fillna(0).astype(int)
user_features['add_to_cart_count'] = user_features['add_to_cart_count'].fillna(0).astype(int)
user_features['purchase_count'] = user_features['purchase_count'].fillna(0).astype(int)

print(f"Event type counts merged with user features")

# Recency: days since last event (from reference time)
user_features['recency_days'] = (REFERENCE_TIME - user_features['last_event_ts']).dt.total_seconds() / 86400

print(f"Recency feature computed")
print(f"\nUser Features Shape: {user_features.shape}")
print(f"Features: {list(user_features.columns)}")
print(f"\nSample user features:")
print(user_features.head(10))

Engineering user-level features

Basic user aggregations computed: (1407580, 5)
Event type counts computed
Event type counts merged with user features
Recency feature computed

User Features Shape: (1407580, 9)
Features: ['user_id', 'total_events', 'unique_products_interacted', 'unique_sessions', 'last_event_ts', 'add_to_cart_count', 'purchase_count', 'views_count', 'recency_days']

Sample user features:
   user_id  total_events  unique_products_interacted  unique_sessions  \
0        0             3                           3                1   
1        1             1                           1                1   
2       10             1                           1                1   
3      100             4                           1                1   
4     1000             1                           1                1   
5    10000             7                           2                2   
6   100000             1                           1                1   
7  10000

---

## 5. Item-Level Feature Engineering

Create one row per product with aggregate features capturing popularity and quality signals.

**Features to create:**
- `total_views`: Number of view events for this product
- `total_add_to_cart`: Number of add-to-cart events
- `total_purchases`: Number of purchase events
- `popularity_score`: Log-scaled popularity metric
- `conversion_rate`: Purchase conversion rate (purchases / views)
- `last_interaction_ts`: Timestamp of most recent interaction
- `recency_days`: Days since last interaction (from reference time)

In [11]:
print("Engineering item-level features\n")

# Basic aggregations per product
item_features = df_events.groupby('product_id').agg(
    last_interaction_ts=('ts_datetime', 'max')
).reset_index()

print(f"Basic item aggregations computed: {item_features.shape}")

# Event type counts per product
item_event_counts = df_events.groupby(['product_id', 'event_type']).size().unstack(fill_value=0)
item_event_counts = item_event_counts.reset_index()

# Rename columns to match requirements
item_column_mapping = {
    'view': 'total_views',
    'add_to_cart': 'total_add_to_cart',
    'purchase': 'total_purchases'
}

item_event_counts = item_event_counts.rename(columns=item_column_mapping)

# Ensure all required columns exist
for col in ['total_views', 'total_add_to_cart', 'total_purchases']:
    if col not in item_event_counts.columns:
        item_event_counts[col] = 0

# Drop 'click' column if it exists
if 'click' in item_event_counts.columns:
    item_event_counts = item_event_counts.drop(columns=['click'])

print(f"Event type counts computed")

# Merge event counts with item features
item_features = item_features.merge(item_event_counts, on='product_id', how='left')

# Fill any NaN values with 0
item_features['total_views'] = item_features['total_views'].fillna(0).astype(int)
item_features['total_add_to_cart'] = item_features['total_add_to_cart'].fillna(0).astype(int)
item_features['total_purchases'] = item_features['total_purchases'].fillna(0).astype(int)

print(f"Event type counts merged with item features")

# Popularity score: log-scaled total interactions
total_interactions = item_features['total_views'] + item_features['total_add_to_cart'] + item_features['total_purchases']
item_features['popularity_score'] = np.log1p(total_interactions)

print(f"Popularity score computed")

# Conversion rate: purchases / views (avoid division by zero)
item_features['conversion_rate'] = item_features['total_purchases'] / item_features['total_views'].replace(0, np.nan)
item_features['conversion_rate'] = item_features['conversion_rate'].fillna(0)

print(f"Conversion rate computed")

# Recency: days since last interaction (from reference time)
item_features['recency_days'] = (REFERENCE_TIME - item_features['last_interaction_ts']).dt.total_seconds() / 86400

print(f"Recency feature computed")
print(f"\nItem Features Shape: {item_features.shape}")
print(f"Features: {list(item_features.columns)}")
print(f"\nSample item features:")
print(item_features.head(10))

Engineering item-level features

Basic item aggregations computed: (235061, 2)
Event type counts computed
Event type counts merged with item features
Popularity score computed
Conversion rate computed
Recency feature computed

Item Features Shape: (235061, 8)
Features: ['product_id', 'last_interaction_ts', 'total_add_to_cart', 'total_purchases', 'total_views', 'popularity_score', 'conversion_rate', 'recency_days']

Sample item features:
  product_id       last_interaction_ts  total_add_to_cart  total_purchases  \
0       1000 2015-07-29 20:05:31+00:00                  0                0   
1     100002 2015-09-05 08:53:54+00:00                  1                0   
2     100004 2015-09-16 03:05:33+00:00                  2                0   
3      10001 2015-09-03 18:07:48+00:00                  0                0   
4     100013 2015-09-09 02:51:21+00:00                  0                0   
5     100014 2015-06-12 00:04:11+00:00                  0                0   
6     100015 

---

## 6. User-Item Interaction Feature Engineering

Create one row per (user_id, product_id) pair capturing interaction strength.

**Features to create:**
- `interaction_count`: Number of interactions between user and product
- `last_interaction_ts`: Timestamp of most recent interaction
- `recency_days`: Days since last interaction (from reference time)
- `has_purchased`: Binary flag (1 if user purchased product, 0 otherwise)

In [12]:
print("Engineering user-item interaction features\n")

# Basic aggregations per user-item pair
interaction_features = df_events.groupby(['user_id', 'product_id']).agg(
    interaction_count=('event_id', 'count'),
    last_interaction_ts=('ts_datetime', 'max')
).reset_index()

print(f"Basic interaction aggregations computed: {interaction_features.shape}")

# Has purchased flag: check if user has any purchase event for this product
purchase_flags = df_events[df_events['event_type'] == 'purchase'].groupby(['user_id', 'product_id']).size().reset_index(name='purchase_event_count')
purchase_flags['has_purchased'] = 1
purchase_flags = purchase_flags[['user_id', 'product_id', 'has_purchased']]

# Merge purchase flags
interaction_features = interaction_features.merge(purchase_flags, on=['user_id', 'product_id'], how='left')
interaction_features['has_purchased'] = interaction_features['has_purchased'].fillna(0).astype(int)

print(f"Purchase flags computed")

# Recency: days since last interaction (from reference time)
interaction_features['recency_days'] = (REFERENCE_TIME - interaction_features['last_interaction_ts']).dt.total_seconds() / 86400

print(f"Recency feature computed")
print(f"\nInteraction Features Shape: {interaction_features.shape}")
print(f"Features: {list(interaction_features.columns)}")
print(f"\nSample interaction features:")
print(interaction_features.head(10))

Engineering user-item interaction features

Basic interaction aggregations computed: (2145179, 4)
Purchase flags computed
Recency feature computed

Interaction Features Shape: (2145179, 6)
Features: ['user_id', 'product_id', 'interaction_count', 'last_interaction_ts', 'has_purchased', 'recency_days']

Sample interaction features:
  user_id product_id  interaction_count       last_interaction_ts  \
0       0     285930                  1 2015-09-11 20:49:49+00:00   
1       0     357564                  1 2015-09-11 20:52:39+00:00   
2       0      67045                  1 2015-09-11 20:55:17+00:00   
3       1      72028                  1 2015-08-13 17:46:06+00:00   
4      10     248766                  1 2015-08-04 18:30:29+00:00   
5     100      36054                  4 2015-09-08 23:52:54+00:00   
6    1000     248975                  1 2015-07-05 13:38:36+00:00   
7   10000     359491                  4 2015-07-03 02:05:41+00:00   
8   10000     401285                  3 2015-07

---

## 7. Light Sanity Checks

Quick validation to ensure features are correctly computed.

In [13]:
print("Feature Sanity Checks\n")

# Check 1: Feature table shapes
print("1. FEATURE TABLE SHAPES:")
print(f"   User features: {user_features.shape}")
print(f"   Item features: {item_features.shape}")
print(f"   Interaction features: {interaction_features.shape}")
print(f"\n   Expected counts:")
print(f"   Users: {df_events['user_id'].nunique():,}")
print(f"   Products: {df_events['product_id'].nunique():,}")
print(f"   User-Product pairs: {df_events.groupby(['user_id', 'product_id']).ngroups:,}")

# Check 2: Display head of each table
print("\n2. USER FEATURES (first 5 rows):")
print(user_features.head())

print("\n3. ITEM FEATURES (first 5 rows):")
print(item_features.head())

print("\n4. INTERACTION FEATURES (first 5 rows):")
print(interaction_features.head())

# Check 3: Ensure no NaNs in critical fields
print("\n5. NULL VALUE CHECK:")
print("\n   User features:")
user_nulls = user_features.isnull().sum()
if user_nulls.sum() == 0:
    print("   No null values")
else:
    print("   Null values found:")
    print(user_nulls[user_nulls > 0])

print("\n   Item features:")
item_nulls = item_features.isnull().sum()
if item_nulls.sum() == 0:
    print("   No null values")
else:
    print("   Null values found:")
    print(item_nulls[item_nulls > 0])

print("\n   Interaction features:")
interaction_nulls = interaction_features.isnull().sum()
if interaction_nulls.sum() == 0:
    print("   No null values")
else:
    print("   Null values found:")
    print(interaction_nulls[interaction_nulls > 0])

print("\nSanity checks complete")

Feature Sanity Checks

1. FEATURE TABLE SHAPES:
   User features: (1407580, 9)
   Item features: (235061, 8)
   Interaction features: (2145179, 6)

   Expected counts:
   Users: 1,407,580
   Products: 235,061
   User-Product pairs: 2,145,179

2. USER FEATURES (first 5 rows):
  user_id  total_events  unique_products_interacted  unique_sessions  \
0       0             3                           3                1   
1       1             1                           1                1   
2      10             1                           1                1   
3     100             4                           1                1   
4    1000             1                           1                1   

              last_event_ts  add_to_cart_count  purchase_count  views_count  \
0 2015-09-11 20:55:17+00:00                  0               0            3   
1 2015-08-13 17:46:06+00:00                  0               0            1   
2 2015-08-04 18:30:29+00:00                  0        

---

## 8. Save Feature Artifacts

Save feature tables to mode-specific directory.

In [14]:
# Define output directory based on DATA_MODE
OUTPUT_DIR = Path(f'artifacts/features/{DATA_MODE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving {DATA_MODE} features to: {OUTPUT_DIR}\n")

# Save user features
user_features_path = OUTPUT_DIR / 'user_features.parquet'
user_features.to_parquet(user_features_path, index=False, compression='snappy')
print(f"Saved user features: {user_features_path}")
print(f"Rows: {len(user_features):,}, Columns: {len(user_features.columns)}")
print(f"File size: {user_features_path.stat().st_size / 1024:.2f} KB")

# Save item features
item_features_path = OUTPUT_DIR / 'item_features.parquet'
item_features.to_parquet(item_features_path, index=False, compression='snappy')
print(f"\nSaved item features: {item_features_path}")
print(f"Rows: {len(item_features):,}, Columns: {len(item_features.columns)}")
print(f"File size: {item_features_path.stat().st_size / 1024:.2f} KB")

# Save interaction features
interaction_features_path = OUTPUT_DIR / 'interaction_features.parquet'
interaction_features.to_parquet(interaction_features_path, index=False, compression='snappy')
print(f"\nSaved interaction features: {interaction_features_path}")
print(f"Rows: {len(interaction_features):,}, Columns: {len(interaction_features.columns)}")
print(f"File size: {interaction_features_path.stat().st_size / 1024:.2f} KB")

Saving retailrocket features to: artifacts\features\retailrocket

Saved user features: artifacts\features\retailrocket\user_features.parquet
Rows: 1,407,580, Columns: 9
File size: 30082.75 KB

Saved item features: artifacts\features\retailrocket\item_features.parquet
Rows: 235,061, Columns: 8
File size: 5740.06 KB

Saved interaction features: artifacts\features\retailrocket\interaction_features.parquet
Rows: 2,145,179, Columns: 6
File size: 50330.56 KB


In [15]:
# Create feature schema documentation
feature_schema = {
 "data_mode": DATA_MODE,
 "reference_time": str(REFERENCE_TIME),
 "description": f"Feature tables for recommender system training - {DATA_MODE} mode",
 "user_features": {
 "description": "Aggregate features per user capturing behavior patterns",
 "key": "user_id",
 "row_count": len(user_features),
 "features": {
 "user_id": "Unique user identifier",
 "total_events": "Total number of events generated by user",
 "unique_products_interacted": "Number of unique products user has interacted with",
 "unique_sessions": "Number of unique sessions for user",
 "last_event_ts": "Timestamp of user's most recent event",
 "views_count": "Number of view events",
 "add_to_cart_count": "Number of add-to-cart events",
 "purchase_count": "Number of purchase events",
 "recency_days": "Days since last event (measured from reference time)"
 }
 },
 "item_features": {
 "description": "Aggregate features per product capturing popularity and quality",
 "key": "product_id",
 "row_count": len(item_features),
 "features": {
 "product_id": "Unique product identifier",
 "last_interaction_ts": "Timestamp of most recent interaction with product",
 "total_views": "Number of view events for this product",
 "total_add_to_cart": "Number of add-to-cart events for this product",
 "total_purchases": "Number of purchase events for this product",
 "popularity_score": "Log-scaled popularity metric (log1p of total interactions)",
 "conversion_rate": "Purchase conversion rate (total_purchases / total_views)",
 "recency_days": "Days since last interaction (measured from reference time)"
 }
 },
 "interaction_features": {
 "description": "Features for each user-item pair capturing interaction strength",
 "key": ["user_id", "product_id"],
 "row_count": len(interaction_features),
 "features": {
 "user_id": "User identifier",
 "product_id": "Product identifier",
 "interaction_count": "Number of interactions between user and product",
 "last_interaction_ts": "Timestamp of most recent interaction between user and product",
 "has_purchased": "Binary flag: 1 if user has purchased product, 0 otherwise",
 "recency_days": "Days since last interaction (measured from reference time)"
 }
 }
}

# Save schema as JSON
schema_path = OUTPUT_DIR / 'feature_schema.json'
with open(schema_path, 'w') as f:
 json.dump(feature_schema, f, indent=2)

print(f"\n Saved feature schema: {schema_path}")
print(f" File size: {schema_path.stat().st_size / 1024:.2f} KB")
print(f"\n Schema includes:")
print(f" - Data mode: {DATA_MODE}")
print(f" - Reference time: {REFERENCE_TIME}")
print(f" - User features: {len(user_features)} rows, {len(user_features.columns)} columns")
print(f" - Item features: {len(item_features)} rows, {len(item_features.columns)} columns")
print(f" - Interaction features: {len(interaction_features)} rows, {len(interaction_features.columns)} columns")


 Saved feature schema: artifacts\features\retailrocket\feature_schema.json
 File size: 2.33 KB

 Schema includes:
 - Data mode: retailrocket
 - Reference time: 2015-09-18 02:59:47+00:00
 - User features: 1407580 rows, 9 columns
 - Item features: 235061 rows, 8 columns
 - Interaction features: 2145179 rows, 6 columns
